<a href="https://colab.research.google.com/github/osergioribeirof/Python/blob/main/SR_GammaFlip__Interativo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
### Rodar essa célula somente uma vez ###
# Delete a # na linha abaixo, execute e coloque de volta a #

#!pip install plotly

In [2]:
import pandas as pd
import plotly
pd.set_option('plotting.backend','plotly')
import plotly.graph_objs as go
import numpy as np
import scipy
from scipy.stats import norm
#import matplotlib.pyplot as plt
import calendar
from datetime import datetime, timedelta, date

In [3]:
pd.options.display.float_format = '{:,.4f}'.format

In [4]:
# Parametros de entrada
filename = 'quotedata.csv'

# Black-Scholes European-Options Gamma
def calcGammaEx(S, K, vol, T, r, q, optType, OI):
    if T == 0 or vol == 0:
        return 0

    dp = (np.log(S/K) + (r - q + 0.5*vol**2)*T) / (vol*np.sqrt(T))
    dm = dp - vol*np.sqrt(T)

    if optType == 'call':
        gamma = np.exp(-q*T) * norm.pdf(dp) / (S * vol * np.sqrt(T))
        return OI * 100 * S * S * 0.01 * gamma
    else: # Gamma is same for calls and puts. This is just to cross-check
        gamma = K * np.exp(-r*T) * norm.pdf(dm) / (S * S * vol * np.sqrt(T))
        return OI * 100 * S * S * 0.01 * gamma

def isThirdFriday(d):
    return d.weekday() == 4 and 15 <= d.day <= 21

In [5]:
# Isso assume que o formato do arquivo CBOE não foi editado, ou seja, a tabela começa na linha 4
optionsFile = open(filename)
optionsFileData = optionsFile.readlines()
optionsFile.close()

In [6]:
# Extraindo SPX spot
spotLine = optionsFileData[1]
spotPrice = float(spotLine.split('Last:')[1].split(',')[0])
fromStrike = 0.8 * spotPrice
toStrike = 1.2 * spotPrice

In [7]:
# Extraindo a data de hoje
dateLine = optionsFileData[2]
todayDate = dateLine.split('Date: ')[1].split(',')
monthDay = todayDate[0].split(' ')

In [8]:
if len(monthDay) == 2:
    year = int(monthDay[4])
    month = monthDay[2]
    day = int(monthDay[0])
else:
    if monthDay[2].isdigit():
        year = int(monthDay[4])
        month = monthDay[2]
        day = int(monthDay[0])
    else:
        year = int(monthDay[4])
        month = monthDay[2]
        day = int(monthDay[0])


# criar um dicionário para mapear os nomes dos meses em português para os equivalentes em inglês
nomes_meses = {'janeiro': 'January', 'fevereiro': 'February', 'março': 'March',
               'abril': 'April', 'maio': 'May', 'junho': 'June',
               'julho': 'July', 'agosto': 'August', 'setembro': 'September',
               'outubro': 'October', 'novembro': 'November', 'dezembro': 'December'}

# extrair o nome do mês da string de entrada
nome_mes_pt = monthDay[2]

# converter o nome do mês para inglês usando o dicionário
nome_mes_en = nomes_meses[nome_mes_pt]

# converter o nome do mês para o número correspondente (por exemplo, 'March' -> 3)
num_mes = datetime.strptime(nome_mes_en, '%B').month

# criar o objeto datetime
todayDate = datetime(year=year, month=num_mes, day=day)

In [9]:
# create a dictionary to map Portuguese month names to English month names
month_names = {'janeiro': 'January', 'fevereiro': 'February', 'março': 'March',
               'abril': 'April', 'maio': 'May', 'junho': 'June',
               'julho': 'July', 'agosto': 'August', 'setembro': 'September',
               'outubro': 'October', 'novembro': 'November', 'dezembro': 'December'}

# extract the month name from the input string
month_name_pt = monthDay[2]

# convert the month name to English using the dictionary
month_name_en = month_names[month_name_pt]

# convert the month name to its corresponding number (e.g., 'March' -> 3)
month_number = datetime.strptime(month_name_en, '%B').month

# create the datetime object
todayDate = datetime(year=year, month=month_number, day=day)

In [10]:
# Get SPX Options Data
df = pd.read_csv(filename, sep=",", header=None, skiprows=4)
df.columns = ['ExpirationDate','Calls','CallLastSale','CallNet','CallBid','CallAsk','CallVol',
              'CallIV','CallDelta','CallGamma','CallOpenInt','StrikePrice','Puts','PutLastSale',
              'PutNet','PutBid','PutAsk','PutVol','PutIV','PutDelta','PutGamma','PutOpenInt']


df['ExpirationDate'] = pd.to_datetime(df['ExpirationDate'], format='%a %b %d %Y')
df['ExpirationDate'] = df['ExpirationDate'] + timedelta(hours=16)
df['StrikePrice'] = df['StrikePrice'].astype(float)
df['CallIV'] = df['CallIV'].astype(float)
df['PutIV'] = df['PutIV'].astype(float)
df['CallGamma'] = df['CallGamma'].astype(float)
df['PutGamma'] = df['PutGamma'].astype(float)
df['CallOpenInt'] = df['CallOpenInt'].astype(float)
df['PutOpenInt'] = df['PutOpenInt'].astype(float)

In [11]:
# ---=== CALCULATE SPOT GAMMA ===---
# Gamma Exposure = Unit Gamma * Open Interest * Contract Size * Spot Price
# To further convert into 'per 1% move' quantity, multiply by 1% of spotPrice
df['CallGEX'] = df['CallGamma'] * df['CallOpenInt'] * 100 * spotPrice * spotPrice * 0.01
df['PutGEX'] = df['PutGamma'] * df['PutOpenInt'] * 100 * spotPrice * spotPrice * 0.01 * -1

df['TotalGamma'] = (df.CallGEX + df.PutGEX) / 10**9
dfAgg = df.groupby(['StrikePrice']).sum(numeric_only=True)
strikes = dfAgg.index.values

In [13]:
# Chart 1: Absolute Gamma Exposure
# define os dados
x_data = strikes
y_data = dfAgg['TotalGamma'].to_numpy()

# cria um gráfico de barras
fig = go.Figure(
    go.Bar(
        x=x_data,
        y=y_data,
        width=6,
        marker_color='rgb(26, 118, 255)',  # cor das barras
        marker_line_color='black',  # cor das linhas de contorno das barras
        marker_line_width=0.15,  # largura das linhas de contorno das barras
        name='Gamma Exposure'
    )
)

# adiciona uma linha vertical para o preço spot
fig.add_shape(
    type='line',
    x0=spotPrice,
    y0=min(y_data),
    x1=spotPrice,
    y1=max(y_data),
    line=dict(
        color='red',
        width=2,
        dash='dash'  # estilo da linha
    )
)

# define o layout do gráfico
fig.update_layout(
    title={
        'text': f"Total Gamma: ${df['TotalGamma'].sum():,.2f} Bn per 1% Ativo Move",
        'font': {'size': 20, 'family': 'Arial Black',}
    },
    xaxis_title='Strike',
    yaxis_title='Spot Gamma Exposure ($ billions/1% move)',
    xaxis=dict(range=[fromStrike, toStrike]),
    yaxis=dict(tickformat='$,.2f'),
    plot_bgcolor='white',  # cor de fundo do gráfico
    font=dict(family='Arial', size=12, color='black')
)

# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig.update_layout(
    width=1750,
    height=800
)


# mostra o gráfico
fig.show()

In [14]:
# DADOS DO CHART 1
dfAgg_sorted = dfAgg.sort_values(by='TotalGamma')

# Get the 3 smallest gamma values
smallest_gamma = dfAgg_sorted.head(3)
print("Menores valores de Gamma:")
print(f"  Put Wall: {smallest_gamma.iloc[0]['TotalGamma']:.4f} at strike {smallest_gamma.index[0]:.0f}")
print(f"  Large Gamma: {smallest_gamma.iloc[1]['TotalGamma']:.4f} at strike {smallest_gamma.index[1]:.0f}")
print(f"  Large Gamma: {smallest_gamma.iloc[2]['TotalGamma']:.4f} at strike {smallest_gamma.index[2]:.0f}")


# Get the 3 largest gamma values
largest_gamma = dfAgg_sorted.tail(3)
print("\nMaiores valores de Gamma:")
print(f"  Call Wall: {largest_gamma.iloc[2]['TotalGamma']:.4f} at strike {largest_gamma.index[2]:.0f}")
print(f"  Large Gamma: {largest_gamma.iloc[1]['TotalGamma']:.4f} at strike {largest_gamma.index[1]:.0f}")
print(f"  Large Gamma: {largest_gamma.iloc[0]['TotalGamma']:.4f} at strike {largest_gamma.index[0]:.0f}")

# Find the positive gamma strike closest to the spot price (Vol Trigger)
positive_gamma_strikes = dfAgg[dfAgg['TotalGamma'] > 0].index
if len(positive_gamma_strikes) > 0:
    # Find the strike closest to the spot price among the positive gamma strikes
    vol_trigger_strike = positive_gamma_strikes[np.abs(positive_gamma_strikes - spotPrice).argmin()]
    vol_trigger_gamma = dfAgg.loc[vol_trigger_strike, 'TotalGamma']
    print(f"\nVol Trigger: {vol_trigger_gamma:.4f} at strike {vol_trigger_strike:.0f}")
else:
    print("\nNo Vol Trigger found (no positive gamma strikes).")


# Print the total gamma
print(f"\nTotal Gamma: ${df['TotalGamma'].sum():,.2f} Bn per 1% SPX Move")

Menores valores de Gamma:
  Put Wall: -0.9442 at strike 660
  Large Gamma: -0.6369 at strike 650
  Large Gamma: -0.5217 at strike 640

Maiores valores de Gamma:
  Call Wall: 2.1126 at strike 673
  Large Gamma: 1.6175 at strike 675
  Large Gamma: 1.4466 at strike 672

Vol Trigger: 1.4466 at strike 672

Total Gamma: $5.28 Bn per 1% SPX Move


In [15]:
# Chart 2: Absolute Gamma Exposure by Calls and Puts
fig = go.Figure()
fig.add_bar(x=strikes, y=dfAgg['CallGEX'].to_numpy() / 10**9, width=6, name="Call Gamma")
fig.add_bar(x=strikes, y=dfAgg['PutGEX'].to_numpy() / 10**9, width=6, name="Put Gamma")
fig.update_xaxes(range=[fromStrike, toStrike])
chartTitle = "Total Gamma: $" + str("{:.2f}".format(df['TotalGamma'].sum())) + " Bn per 1% SPX Move"
fig.update_layout(title_text=chartTitle, title_font=dict(size=20, family="Arial Black"))
fig.update_xaxes(title_text="Strike")
fig.update_yaxes(title_text="Spot Gamma Exposure ($ billions/1% move)")
fig.add_shape(dict(type="line", x0=spotPrice, y0=0, x1=spotPrice, y1=max(dfAgg['CallGEX'].to_numpy() / 10**9), line=dict(color="black", width=2), name="SPX Spot:" + str("{:,.0f}".format(spotPrice))))

# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig.update_layout(
    width=1750,
    height=800
)

fig.show()


In [16]:
# DADOS CHART 2
dfAgg['AbsoluteTotalGEX'] = dfAgg['CallGEX'].abs() + dfAgg['PutGEX'].abs()

# Sort by AbsoluteTotalGEX to find the strikes with the largest combined exposure
dfAgg_sorted_gex = dfAgg.sort_values(by='AbsoluteTotalGEX', ascending=False)

# Get the top 6 strikes based on combined absolute GEX
gex_levels = dfAgg_sorted_gex.head(6)

print("Top 6 GEX Levels (based on sum of absolute Call and Put Gamma Exposure):")
for i in range(len(gex_levels)):
    strike = gex_levels.index[i]
    call_gex = gex_levels.iloc[i]['CallGEX'] / 10**9
    put_gex = gex_levels.iloc[i]['PutGEX'] / 10**9
    print(f"  GEX Level {i+1}: Strike {strike:.0f}, Call GEX: {call_gex:.4f} Bn, Put GEX: {put_gex:.4f} Bn")

Top 6 GEX Levels (based on sum of absolute Call and Put Gamma Exposure):
  GEX Level 1: Strike 672, Call GEX: 2.3966 Bn, Put GEX: -0.9500 Bn
  GEX Level 2: Strike 670, Call GEX: 1.8853 Bn, Put GEX: -1.4222 Bn
  GEX Level 3: Strike 673, Call GEX: 2.6211 Bn, Put GEX: -0.5085 Bn
  GEX Level 4: Strike 675, Call GEX: 2.0713 Bn, Put GEX: -0.4538 Bn
  GEX Level 5: Strike 671, Call GEX: 1.7471 Bn, Put GEX: -0.7303 Bn
  GEX Level 6: Strike 660, Call GEX: 0.5586 Bn, Put GEX: -1.5028 Bn


In [17]:
# ---=== CALCULATE GAMMA PROFILE ===---
levels = np.linspace(fromStrike, toStrike, 60)

# For 0DTE options, I'm setting DTE = 1 day, otherwise they get excluded
df['daysTillExp'] = [1/262 if (np.busday_count(todayDate.date(), x.date())) == 0 \
                           else np.busday_count(todayDate.date(), x.date())/262 for x in df.ExpirationDate]

nextExpiry = df['ExpirationDate'].min()

df['IsThirdFriday'] = [isThirdFriday(x) for x in df.ExpirationDate]
thirdFridays = df.loc[df['IsThirdFriday'] == True]
nextMonthlyExp = thirdFridays['ExpirationDate'].min()

totalGamma = []
totalGammaExNext = []
totalGammaExFri = []

In [18]:
# For each spot level, calc gamma exposure at that point
for level in levels:
    df['callGammaEx'] = df.apply(lambda row : calcGammaEx(level, row['StrikePrice'], row['CallIV'],
                                                          row['daysTillExp'], 0, 0, "call", row['CallOpenInt']), axis = 1)

    df['putGammaEx'] = df.apply(lambda row : calcGammaEx(level, row['StrikePrice'], row['PutIV'],
                                                         row['daysTillExp'], 0, 0, "put", row['PutOpenInt']), axis = 1)

    totalGamma.append(df['callGammaEx'].sum() - df['putGammaEx'].sum())

    exNxt = df.loc[df['ExpirationDate'] != nextExpiry]
    totalGammaExNext.append(exNxt['callGammaEx'].sum() - exNxt['putGammaEx'].sum())

    exFri = df.loc[df['ExpirationDate'] != nextMonthlyExp]
    totalGammaExFri.append(exFri['callGammaEx'].sum() - exFri['putGammaEx'].sum())

totalGamma = np.array(totalGamma) / 10**9
totalGammaExNext = np.array(totalGammaExNext) / 10**9
totalGammaExFri = np.array(totalGammaExFri) / 10**9

In [19]:
# Find Gamma Flip Point
zeroCrossIdx = np.where(np.diff(np.sign(totalGamma)))[0]

negGamma = totalGamma[zeroCrossIdx]
posGamma = totalGamma[zeroCrossIdx+1]
negStrike = levels[zeroCrossIdx]
posStrike = levels[zeroCrossIdx+1]

zeroGamma = posStrike - ((posStrike - negStrike) * posGamma/(posGamma-negGamma))
zeroGamma = zeroGamma[0]

In [20]:
# Chart 3: Gamma Exposure Profile
fig = go.Figure()

fig.add_trace(go.Scatter(x=levels, y=totalGamma, mode='lines', name='All Expiries'))
fig.add_trace(go.Scatter(x=levels, y=totalGammaExNext, mode='lines', name='Ex-Next Expiry'))
fig.add_trace(go.Scatter(x=levels, y=totalGammaExFri, mode='lines', name='Ex-Next Monthly Expiry'))

chartTitle = "Gamma Exposure Profile, SPX, " + todayDate.strftime('%d %b %Y')
fig.update_layout(title=chartTitle, xaxis_title='Index Price', yaxis_title='Gamma Exposure ($ billions/1% move)')
fig.update_layout(title_text=chartTitle, title_font=dict(size=20, family="Arial Black"))

fig.add_shape(
    dict(
        type="line",
        x0=spotPrice,
        y0=min(totalGamma),
        x1=spotPrice,
        y1=max(totalGamma),
        line=dict(color="red", width=1.5),
        name="SPX Spot: " + str("{:,.0f}".format(spotPrice))
    )
)

fig.add_shape(
    dict(
        type="line",
        x0=zeroGamma,
        y0=min(totalGamma),
        x1=zeroGamma,
        y1=max(totalGamma),
        line=dict(color="green", width=1.5),
        name="Gamma Flip: " + str("{:,.0f}".format(zeroGamma))
    )
)

fig.update_xaxes(range=[fromStrike, toStrike])
fig.update_yaxes(range=[min(totalGamma), max(totalGamma)])

fig.add_trace(
    go.Scatter(
        x=[fromStrike, zeroGamma, toStrike],
        y=[min(totalGamma), min(totalGamma), min(totalGamma)],
        mode="none",
        fill="toself",
        fillcolor="red",
        opacity=0.1,
        showlegend=False,
        name="Negative Gamma"
    )
)

fig.add_trace(
    go.Scatter(
        x=[fromStrike, zeroGamma, toStrike],
        y=[max(totalGamma), max(totalGamma), max(totalGamma)],
        mode="none",
        fill="toself",
        fillcolor="green",
        opacity=0.1,
        showlegend=False,
        name="Positive Gamma"
    )
)

# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig.update_layout(
    width=1400,
    height=700
)

fig.show()

In [21]:
# DADOS CHART 3
# Gamma Flip (primeiro valor acima da linha verde central)
gamma_flip_index = np.where(totalGammaExFri > 0)[0][0] if np.any(totalGammaExFri > 0) else None
if gamma_flip_index is not None:
    gamma_flip_strike = levels[gamma_flip_index]
    gamma_flip_value = totalGammaExFri[gamma_flip_index]
    print(f"Gamma Flip (Ex-Next Monthly Expiry): {gamma_flip_value:.4f} at strike {gamma_flip_strike:.0f}")
else:
    print("No Gamma Flip found (Ex-Next Monthly Expiry).")


# Max Gamma Positivo
max_gamma_positive_value = np.max(totalGammaExFri)
max_gamma_positive_index = np.argmax(totalGammaExFri)
max_gamma_positive_strike = levels[max_gamma_positive_index]
print(f"Max Gamma Positivo (Ex-Next Monthly Expiry): {max_gamma_positive_value:.4f} at strike {max_gamma_positive_strike:.0f}")


# Min Gamma Negativo
min_gamma_negative_value = np.min(totalGammaExFri)
min_gamma_negative_index = np.argmin(totalGammaExFri)
min_gamma_negative_strike = levels[min_gamma_negative_index]
print(f"Min Gamma Negativo (Ex-Next Monthly Expiry): {min_gamma_negative_value:.4f} at strike {min_gamma_negative_strike:.0f}")

Gamma Flip (Ex-Next Monthly Expiry): 0.0556 at strike 670
Max Gamma Positivo (Ex-Next Monthly Expiry): 7.7053 at strike 684
Min Gamma Negativo (Ex-Next Monthly Expiry): -12.0315 at strike 647


In [22]:
# Consolidating results from CHART 1, CHART 2, and CHART 3

print("--- DADOS CHART 1 ---")
# DADOS CHART 1
dfAgg_sorted = dfAgg.sort_values(by='TotalGamma')

# Get the 3 smallest gamma values
smallest_gamma = dfAgg_sorted.head(3)
print("Menores valores de Gamma:")
print(f"  Put Wall: {smallest_gamma.iloc[0]['TotalGamma']:.4f} at strike {smallest_gamma.index[0]:.0f}")
print(f"  Large Gamma: {smallest_gamma.iloc[1]['TotalGamma']:.4f} at strike {smallest_gamma.index[1]:.0f}")
print(f"  Large Gamma: {smallest_gamma.iloc[2]['TotalGamma']:.4f} at strike {smallest_gamma.index[2]:.0f}")


# Get the 3 largest gamma values
largest_gamma = dfAgg_sorted.tail(3)
print("\nMaiores valores de Gamma:")
print(f"  Call Wall: {largest_gamma.iloc[2]['TotalGamma']:.4f} at strike {largest_gamma.index[2]:.0f}")
print(f"  Large Gamma: {largest_gamma.iloc[1]['TotalGamma']:.4f} at strike {largest_gamma.index[1]:.0f}")
print(f"  Large Gamma: {largest_gamma.iloc[0]['TotalGamma']:.4f} at strike {largest_gamma.index[0]:.0f}")

# Find the positive gamma strike closest to the spot price (Vol Trigger)
positive_gamma_strikes = dfAgg[dfAgg['TotalGamma'] > 0].index
if len(positive_gamma_strikes) > 0:
    # Find the strike closest to the spot price among the positive gamma strikes
    vol_trigger_strike = positive_gamma_strikes[np.abs(positive_gamma_strikes - spotPrice).argmin()]
    vol_trigger_gamma = dfAgg.loc[vol_trigger_strike, 'TotalGamma']
    print(f"\nVol Trigger: {vol_trigger_gamma:.4f} at strike {vol_trigger_strike:.0f}")
else:
    print("\nNo Vol Trigger found (no positive gamma strikes).")


# Print the total gamma
print(f"\nTotal Gamma: ${df['TotalGamma'].sum():,.2f} Bn per 1% SPX Move")

print("\n--- DADOS CHART 2 ---")
# DADOS CHART 2
dfAgg['AbsoluteTotalGEX'] = dfAgg['CallGEX'].abs() + dfAgg['PutGEX'].abs()

# Sort by AbsoluteTotalGEX to find the strikes with the largest combined exposure
dfAgg_sorted_gex = dfAgg.sort_values(by='AbsoluteTotalGEX', ascending=False)

# Get the top 6 strikes based on combined absolute GEX
gex_levels = dfAgg_sorted_gex.head(6)

print("Top 6 GEX Levels (based on sum of absolute Call and Put Gamma Exposure):")
for i in range(len(gex_levels)):
    strike = gex_levels.index[i]
    call_gex = gex_levels.iloc[i]['CallGEX'] / 10**9
    put_gex = gex_levels.iloc[i]['PutGEX'] / 10**9
    print(f"  GEX Level {i+1}: Strike {strike:.0f}, Call GEX: {call_gex:.4f} Bn, Put GEX: {put_gex:.4f} Bn")

print("\n--- DADOS CHART 3 ---")
# DADOS CHART 3
# Gamma Flip (primeiro valor acima da linha verde central)
gamma_flip_index = np.where(totalGammaExFri > 0)[0][0] if np.any(totalGammaExFri > 0) else None
if gamma_flip_index is not None:
    gamma_flip_strike = levels[gamma_flip_index]
    gamma_flip_value = totalGammaExFri[gamma_flip_index]
    print(f"Gamma Flip (Ex-Next Monthly Expiry): {gamma_flip_value:.4f} at strike {gamma_flip_strike:.0f}")
else:
    print("No Gamma Flip found (Ex-Next Monthly Expiry).")


# Max Gamma Positivo
max_gamma_positive_value = np.max(totalGammaExFri)
max_gamma_positive_index = np.argmax(totalGammaExFri)
max_gamma_positive_strike = levels[max_gamma_positive_index]
print(f"Max Gamma Positivo (Ex-Next Monthly Expiry): {max_gamma_positive_value:.4f} at strike {max_gamma_positive_strike:.0f}")


# Min Gamma Negativo
min_gamma_negative_value = np.min(totalGammaExFri)
min_gamma_negative_index = np.argmin(totalGammaExFri)
min_gamma_negative_strike = levels[min_gamma_negative_index]
print(f"Min Gamma Negativo (Ex-Next Monthly Expiry): {min_gamma_negative_value:.4f} at strike {min_gamma_negative_strike:.0f}")

--- DADOS CHART 1 ---
Menores valores de Gamma:
  Put Wall: -0.9442 at strike 660
  Large Gamma: -0.6369 at strike 650
  Large Gamma: -0.5217 at strike 640

Maiores valores de Gamma:
  Call Wall: 2.1126 at strike 673
  Large Gamma: 1.6175 at strike 675
  Large Gamma: 1.4466 at strike 672

Vol Trigger: 1.4466 at strike 672

Total Gamma: $5.28 Bn per 1% SPX Move

--- DADOS CHART 2 ---
Top 6 GEX Levels (based on sum of absolute Call and Put Gamma Exposure):
  GEX Level 1: Strike 672, Call GEX: 2.3966 Bn, Put GEX: -0.9500 Bn
  GEX Level 2: Strike 670, Call GEX: 1.8853 Bn, Put GEX: -1.4222 Bn
  GEX Level 3: Strike 673, Call GEX: 2.6211 Bn, Put GEX: -0.5085 Bn
  GEX Level 4: Strike 675, Call GEX: 2.0713 Bn, Put GEX: -0.4538 Bn
  GEX Level 5: Strike 671, Call GEX: 1.7471 Bn, Put GEX: -0.7303 Bn
  GEX Level 6: Strike 660, Call GEX: 0.5586 Bn, Put GEX: -1.5028 Bn

--- DADOS CHART 3 ---
Gamma Flip (Ex-Next Monthly Expiry): 0.0556 at strike 670
Max Gamma Positivo (Ex-Next Monthly Expiry): 7.7053 

In [23]:
# ==================== CÉLULA FINAL DO NOTEBOOK ====================
# Cole esta célula no final do seu notebook Jupyter/Colab
# Ela irá gerar UMA ÚNICA LINHA para atualizar tudo de uma vez

print("\n" + "="*80)
print("🚀 GERADOR DE CÓDIGO TRADINGVIEW - GAMMA FLIP (VERSÃO SIMPLIFICADA)")
print("="*80 + "\n")

# ==================== COLETA DOS DADOS ====================

# CHART 1 - Spot Gamma Levels
c1_put_wall_value = smallest_gamma.iloc[0]['TotalGamma']
c1_put_wall_strike = smallest_gamma.index[0]
c1_lg1_value = smallest_gamma.iloc[1]['TotalGamma']
c1_lg1_strike = smallest_gamma.index[1]
c1_lg2_value = smallest_gamma.iloc[2]['TotalGamma']
c1_lg2_strike = smallest_gamma.index[2]
c1_call_wall_value = largest_gamma.iloc[2]['TotalGamma']
c1_call_wall_strike = largest_gamma.index[2]
c1_lg3_value = largest_gamma.iloc[1]['TotalGamma']
c1_lg3_strike = largest_gamma.index[1]
c1_lg4_value = largest_gamma.iloc[0]['TotalGamma']
c1_lg4_strike = largest_gamma.index[0]
c1_vol_trigger_value = vol_trigger_gamma
c1_vol_trigger_strike = vol_trigger_strike

# CHART 2 - GEX Levels
c2_gex1_strike = gex_levels.index[0]
c2_gex1_call = gex_levels.iloc[0]['CallGEX'] / 10**9
c2_gex1_put = gex_levels.iloc[0]['PutGEX'] / 10**9

c2_gex2_strike = gex_levels.index[1]
c2_gex2_call = gex_levels.iloc[1]['CallGEX'] / 10**9
c2_gex2_put = gex_levels.iloc[1]['PutGEX'] / 10**9

c2_gex3_strike = gex_levels.index[2]
c2_gex3_call = gex_levels.iloc[2]['CallGEX'] / 10**9
c2_gex3_put = gex_levels.iloc[2]['PutGEX'] / 10**9

c2_gex4_strike = gex_levels.index[3]
c2_gex4_call = gex_levels.iloc[3]['CallGEX'] / 10**9
c2_gex4_put = gex_levels.iloc[3]['PutGEX'] / 10**9

c2_gex5_strike = gex_levels.index[4]
c2_gex5_call = gex_levels.iloc[4]['CallGEX'] / 10**9
c2_gex5_put = gex_levels.iloc[4]['PutGEX'] / 10**9

c2_gex6_strike = gex_levels.index[5]
c2_gex6_call = gex_levels.iloc[5]['CallGEX'] / 10**9
c2_gex6_put = gex_levels.iloc[5]['PutGEX'] / 10**9

# CHART 3 - Gamma Profile
c3_gamma_flip_value = gamma_flip_value
c3_gamma_flip_strike = gamma_flip_strike
c3_max_pos_value = max_gamma_positive_value
c3_max_pos_strike = max_gamma_positive_strike
c3_min_neg_value = min_gamma_negative_value
c3_min_neg_strike = min_gamma_negative_strike

# Dados gerais
spot_price = spotPrice
total_gamma = df['TotalGamma'].sum()
update_date = todayDate.strftime('%d %b %Y 00:00')

# ==================== GERAÇÃO DA LINHA ÚNICA ====================

# Criar a string com todos os dados separados por vírgula
data_string = f"{spot_price},{total_gamma},{update_date},"
data_string += f"{c1_put_wall_value},{c1_put_wall_strike},"
data_string += f"{c1_lg1_value},{c1_lg1_strike},"
data_string += f"{c1_lg2_value},{c1_lg2_strike},"
data_string += f"{c1_call_wall_value},{c1_call_wall_strike},"
data_string += f"{c1_lg3_value},{c1_lg3_strike},"
data_string += f"{c1_lg4_value},{c1_lg4_strike},"
data_string += f"{c1_vol_trigger_value},{c1_vol_trigger_strike},"
data_string += f"{c2_gex1_strike},{c2_gex1_call},{c2_gex1_put},"
data_string += f"{c2_gex2_strike},{c2_gex2_call},{c2_gex2_put},"
data_string += f"{c2_gex3_strike},{c2_gex3_call},{c2_gex3_put},"
data_string += f"{c2_gex4_strike},{c2_gex4_call},{c2_gex4_put},"
data_string += f"{c2_gex5_strike},{c2_gex5_call},{c2_gex5_put},"
data_string += f"{c2_gex6_strike},{c2_gex6_call},{c2_gex6_put},"
data_string += f"{c3_gamma_flip_value},{c3_gamma_flip_strike},"
data_string += f"{c3_max_pos_value},{c3_max_pos_strike},"
data_string += f"{c3_min_neg_value},{c3_min_neg_strike}"

print("📋 COPIE APENAS ESTA LINHA ABAIXO E COLE NO TRADINGVIEW:\n")
print("="*80)
print(data_string)
print("="*80 + "\n")

# ==================== INSTRUÇÕES ====================

print("💡 COMO USAR:\n")
print("1. ✅ COPIE a linha acima (entre as linhas ====)")
print("2. ✅ Abra o TradingView e o indicador 'SR Gamma Flip - COMPLETO'")
print("3. ✅ Clique no ícone de engrenagem (Settings)")
print("4. ✅ Procure o campo 'Data String' no grupo 'Dados de Entrada'")
print("5. ✅ COLE a linha copiada neste campo")
print("6. ✅ Clique em 'OK'")
print("7. ✅ PRONTO! Todos os dados serão atualizados automaticamente!")

print("\n" + "="*80)
print("📊 RESUMO DOS DADOS EXTRAÍDOS:\n")
print("="*80)

print(f"\n🟢 SPOT PRICE: ${spot_price:,.2f}")
print(f"📊 TOTAL GAMMA: ${total_gamma:.2f} Bn")
print(f"📅 DATA: {update_date}")

print("\n🔴 SUPORTES (Put Levels):")
print(f"   • Put Wall: {c1_put_wall_strike:.0f} (γ: {c1_put_wall_value:.2f})")
print(f"   • Large 1: {c1_lg1_strike:.0f} (γ: {c1_lg1_value:.2f})")
print(f"   • Large 2: {c1_lg2_strike:.0f} (γ: {c1_lg2_value:.2f})")

print("\n🟢 RESISTÊNCIAS (Call Levels):")
print(f"   • Call Wall: {c1_call_wall_strike:.0f} (γ: {c1_call_wall_value:.2f})")
print(f"   • Large 3: {c1_lg3_strike:.0f} (γ: {c1_lg3_value:.2f})")
print(f"   • Large 4: {c1_lg4_strike:.0f} (γ: {c1_lg4_value:.2f})")

print("\n🟠 NÍVEIS ESPECIAIS:")
print(f"   • Vol Trigger: {c1_vol_trigger_strike:.0f}")
print(f"   • Gamma Flip: {c3_gamma_flip_strike:.0f}")

print("\n💎 TOP 3 GEX LEVELS:")
print(f"   1. Strike {c2_gex1_strike:.0f}: Net GEX = {(c2_gex1_call + c2_gex1_put):.2f} Bn")
print(f"   2. Strike {c2_gex2_strike:.0f}: Net GEX = {(c2_gex2_call + c2_gex2_put):.2f} Bn")
print(f"   3. Strike {c2_gex3_strike:.0f}: Net GEX = {(c2_gex3_call + c2_gex3_put):.2f} Bn")

regime = "POSITIVE GAMMA ✅" if spot_price > c3_gamma_flip_strike else "NEGATIVE GAMMA ⚠️"
print(f"\n📈 REGIME: {regime}")

distance_to_flip = ((spot_price - c3_gamma_flip_strike) / c3_gamma_flip_strike) * 100
print(f"📏 DISTÂNCIA DO FLIP: {distance_to_flip:.2f}%")

print("\n" + "="*80)
print("✅ DADOS PRONTOS PARA USAR!")
print("="*80 + "\n")

# ==================== VALIDAÇÃO ====================

print("🔍 VALIDAÇÃO:\n")

errors = []
if spot_price <= 0:
    errors.append("❌ Spot Price inválido")
if c1_put_wall_strike <= 0:
    errors.append("❌ Put Wall inválido")
if c1_call_wall_strike <= 0:
    errors.append("❌ Call Wall inválido")
if c3_gamma_flip_strike <= 0:
    errors.append("❌ Gamma Flip inválido")

if errors:
    print("⚠️ AVISOS:")
    for error in errors:
        print(f"   {error}")
else:
    print("✅ Todos os dados validados!")
    print("✅ String pronta para uso!")

print("\n" + "="*80 + "\n")
# ==================== CÉLULA FINAL DO NOTEBOOK ====================
# Cole esta célula no final do seu notebook Jupyter/Colab
# Ela irá gerar o código TradingView automaticamente

print("\n" + "="*80)
print("🚀 GERADOR DE CÓDIGO TRADINGVIEW - GAMMA FLIP")
print("="*80 + "\n")

# ==================== COLETA DOS DADOS ====================

# CHART 1 - Spot Gamma Levels
chart1_data = {
    'Put Wall': {
        'value': smallest_gamma.iloc[0]['TotalGamma'],
        'strike': smallest_gamma.index[0]
    },
    'Large Gamma 1': {
        'value': smallest_gamma.iloc[1]['TotalGamma'],
        'strike': smallest_gamma.index[1]
    },
    'Large Gamma 2': {
        'value': smallest_gamma.iloc[2]['TotalGamma'],
        'strike': smallest_gamma.index[2]
    },
    'Call Wall': {
        'value': largest_gamma.iloc[2]['TotalGamma'],
        'strike': largest_gamma.index[2]
    },
    'Large Gamma 3': {
        'value': largest_gamma.iloc[1]['TotalGamma'],
        'strike': largest_gamma.index[1]
    },
    'Large Gamma 4': {
        'value': largest_gamma.iloc[0]['TotalGamma'],
        'strike': largest_gamma.index[0]
    },
    'Vol Trigger': {
        'value': vol_trigger_gamma,
        'strike': vol_trigger_strike
    }
}

# CHART 2 - GEX Levels
chart2_data = {}
for i in range(min(6, len(gex_levels))):
    chart2_data[f'GEX Level {i+1}'] = {
        'strike': gex_levels.index[i],
        'call_gex': gex_levels.iloc[i]['CallGEX'] / 10**9,
        'put_gex': gex_levels.iloc[i]['PutGEX'] / 10**9
    }

# CHART 3 - Gamma Profile
chart3_data = {
    'Gamma Flip': {
        'value': gamma_flip_value,
        'strike': gamma_flip_strike
    },
    'Max Gamma Positivo': {
        'value': max_gamma_positive_value,
        'strike': max_gamma_positive_strike
    },
    'Min Gamma Negativo': {
        'value': min_gamma_negative_value,
        'strike': min_gamma_negative_strike
    }
}

# Dados gerais
general_data = {
    'spot_price': spotPrice,
    'total_gamma': df['TotalGamma'].sum(),
    'update_date': todayDate.strftime('%d %b %Y 00:00')
}

# ==================== GERAÇÃO DO CÓDIGO ====================

print("📋 CÓDIGO PARA TRADINGVIEW - COPIE E COLE NOS INPUTS\n")
print("="*80)
print("// Cole este código no indicador TradingView")
print("// Substitua apenas os VALORES dos inputs existentes")
print("="*80 + "\n")

# DADOS GERAIS
print("// ==================== DADOS GERAIS ====================")
print(f"spotPrice = {general_data['spot_price']:.2f}")
print(f"totalGamma = {general_data['total_gamma']:.2f}")
print(f'updateDate = timestamp("{general_data["update_date"]}")')
print()

# CHART 1
print("// ==================== CHART 1: SPOT GAMMA LEVELS ====================")
print(f"c1_put_wall = {chart1_data['Put Wall']['value']:.4f}")
print(f"c1_put_wall_strike = {chart1_data['Put Wall']['strike']:.0f}")
print()
print(f"c1_lg1 = {chart1_data['Large Gamma 1']['value']:.4f}")
print(f"c1_lg1_strike = {chart1_data['Large Gamma 1']['strike']:.0f}")
print()
print(f"c1_lg2 = {chart1_data['Large Gamma 2']['value']:.4f}")
print(f"c1_lg2_strike = {chart1_data['Large Gamma 2']['strike']:.0f}")
print()
print(f"c1_call_wall = {chart1_data['Call Wall']['value']:.4f}")
print(f"c1_call_wall_strike = {chart1_data['Call Wall']['strike']:.0f}")
print()
print(f"c1_lg3 = {chart1_data['Large Gamma 3']['value']:.4f}")
print(f"c1_lg3_strike = {chart1_data['Large Gamma 3']['strike']:.0f}")
print()
print(f"c1_lg4 = {chart1_data['Large Gamma 4']['value']:.4f}")
print(f"c1_lg4_strike = {chart1_data['Large Gamma 4']['strike']:.0f}")
print()
print(f"c1_vol_trigger = {chart1_data['Vol Trigger']['value']:.4f}")
print(f"c1_vol_trigger_strike = {chart1_data['Vol Trigger']['strike']:.0f}")
print()

# CHART 2
print("// ==================== CHART 2: GEX LEVELS ====================")
for key, value in chart2_data.items():
    idx = key.split()[-1]  # Pega o número
    print(f"c2_gex{idx}_strike = {value['strike']:.0f}")
    print(f"c2_gex{idx}_call = {value['call_gex']:.4f}")
    print(f"c2_gex{idx}_put = {value['put_gex']:.4f}")
    print()

# CHART 3
print("// ==================== CHART 3: GAMMA PROFILE ====================")
print(f"c3_gamma_flip = {chart3_data['Gamma Flip']['value']:.4f}")
print(f"c3_gamma_flip_strike = {chart3_data['Gamma Flip']['strike']:.0f}")
print()
print(f"c3_max_pos = {chart3_data['Max Gamma Positivo']['value']:.4f}")
print(f"c3_max_pos_strike = {chart3_data['Max Gamma Positivo']['strike']:.0f}")
print()
print(f"c3_min_neg = {chart3_data['Min Gamma Negativo']['value']:.4f}")
print(f"c3_min_neg_strike = {chart3_data['Min Gamma Negativo']['strike']:.0f}")

print("\n" + "="*80)
print("✅ CÓDIGO GERADO COM SUCESSO!")
print("="*80 + "\n")

# ==================== RESUMO DOS DADOS ====================

print("📊 RESUMO DOS NÍVEIS IMPORTANTES:\n")

print("🔴 SUPORTES (Put Levels):")
print(f"   • Put Wall: {chart1_data['Put Wall']['strike']:.0f} (Gamma: {chart1_data['Put Wall']['value']:.2f})")
print(f"   • Large 1: {chart1_data['Large Gamma 1']['strike']:.0f} (Gamma: {chart1_data['Large Gamma 1']['value']:.2f})")
print(f"   • Large 2: {chart1_data['Large Gamma 2']['strike']:.0f} (Gamma: {chart1_data['Large Gamma 2']['value']:.2f})")

print("\n🟢 RESISTÊNCIAS (Call Levels):")
print(f"   • Call Wall: {chart1_data['Call Wall']['strike']:.0f} (Gamma: {chart1_data['Call Wall']['value']:.2f})")
print(f"   • Large 3: {chart1_data['Large Gamma 3']['strike']:.0f} (Gamma: {chart1_data['Large Gamma 3']['value']:.2f})")
print(f"   • Large 4: {chart1_data['Large Gamma 4']['strike']:.0f} (Gamma: {chart1_data['Large Gamma 4']['value']:.2f})")

print("\n🟠 NÍVEIS ESPECIAIS:")
print(f"   • Vol Trigger: {chart1_data['Vol Trigger']['strike']:.0f}")
print(f"   • Gamma Flip: {chart3_data['Gamma Flip']['strike']:.0f}")

print("\n💎 TOP 3 GEX LEVELS:")
for i in range(min(3, len(chart2_data))):
    key = f'GEX Level {i+1}'
    data = chart2_data[key]
    net_gex = data['call_gex'] + data['put_gex']
    print(f"   {i+1}. Strike {data['strike']:.0f}: Net GEX = {net_gex:.2f} Bn")

print("\n📈 STATUS DO MERCADO:")
print(f"   • Spot Price: ${general_data['spot_price']:,.2f}")
print(f"   • Total Gamma: ${general_data['total_gamma']:.2f} Bn")

regime = "POSITIVE GAMMA ✅" if general_data['spot_price'] > chart3_data['Gamma Flip']['strike'] else "NEGATIVE GAMMA ⚠️"
print(f"   • Regime: {regime}")

distance_to_flip = ((general_data['spot_price'] - chart3_data['Gamma Flip']['strike']) /
                    chart3_data['Gamma Flip']['strike']) * 100
print(f"   • Distância do Flip: {distance_to_flip:.2f}%")

print(f"\n🕐 Última atualização: {general_data['update_date']}")

print("\n" + "="*80)
print("💡 PRÓXIMOS PASSOS:")
print("="*80)
print("1. ✅ Copie os valores acima")
print("2. ✅ Abra o indicador no TradingView")
print("3. ✅ Clique no ícone de engrenagem (Settings)")
print("4. ✅ Cole os valores nos inputs correspondentes")
print("5. ✅ Clique em 'OK' para aplicar")
print("\n💾 Dica: Salve este código em um arquivo .txt para referência futura!")
print("="*80 + "\n")

# ==================== VALIDAÇÃO ====================

print("🔍 VALIDAÇÃO DOS DADOS:\n")

# Verifica se há dados válidos
errors = []

if spotPrice <= 0:
    errors.append("❌ Spot Price inválido")

if not any([v['strike'] > 0 for v in chart1_data.values()]):
    errors.append("❌ Strikes do Chart 1 inválidos")

if not any([v['strike'] > 0 for v in chart2_data.values()]):
    errors.append("❌ Strikes do Chart 2 inválidos")

if not any([v['strike'] > 0 for v in chart3_data.values()]):
    errors.append("❌ Strikes do Chart 3 inválidos")

if errors:
    print("⚠️ AVISOS ENCONTRADOS:")
    for error in errors:
        print(f"   {error}")
else:
    print("✅ Todos os dados validados com sucesso!")
    print("✅ Pronto para usar no TradingView!")

print("\n" + "="*80 + "\n")


🚀 GERADOR DE CÓDIGO TRADINGVIEW - GAMMA FLIP (VERSÃO SIMPLIFICADA)

📋 COPIE APENAS ESTA LINHA ABAIXO E COLE NO TRADINGVIEW:

672.2301,5.275755308251696,23 Oct 2025 00:00,-0.9441562321677914,660.0,-0.636860016442342,650.0,-0.5216892680202082,640.0,2.112575905817385,673.0,1.6174768082200388,675.0,1.4465698104058329,672.0,1.4465698104058329,672.0,672.0,2.396576705223833,-0.9500068948180003,670.0,1.8853193490143767,-1.4221719099848986,673.0,2.6210850728782167,-0.5085091670608315,675.0,2.0712885382237247,-0.453811730003686,671.0,1.7471023582429113,-0.7302884606534916,660.0,0.5586050197586033,-1.5027612519263946,0.055622490129821775,669.9513538983051,7.70530741328966,683.6238305084746,-12.03150553769605,647.1638928813559

💡 COMO USAR:

1. ✅ COPIE a linha acima (entre as linhas ====)
2. ✅ Abra o TradingView e o indicador 'SR Gamma Flip - COMPLETO'
3. ✅ Clique no ícone de engrenagem (Settings)
4. ✅ Procure o campo 'Data String' no grupo 'Dados de Entrada'
5. ✅ COLE a linha copiada neste campo